# 🦆 Retraining Colab — U-Net (3 kelas, anti-collapse) + AWC + Figures

Notebook ini **melatih ulang** pipeline deteksi fertilitas telur bebek di Google Colab:

1. **U-Net 3 kelas** (BG / Vaskular / Embrio) dengan **loss yang benar** (weighted Cross-Entropy + multiclass Dice) untuk mengatasi *class-imbalance collapse* (model lama prediksi background semua).
2. **Ekstraksi fitur** (classical + deep embedding dari U-Net baru).
3. **AWC** (Adaptive Weighted Clustering) dilatih ulang + evaluasi.
4. **Semua figure** dibangun inline supaya bisa langsung dicek (termasuk error AWC).

> ⚠️ **Penyebab collapse model lama:** mask 96% background, embrio ~3%, vaskular <1%. Dengan `loss_type: ce` polos, model menang cukup menebak background. Notebook ini memakai **class weight + Dice** untuk memaksa model belajar kelas minoritas.

**Urutan:** jalankan sel dari atas ke bawah. Aktifkan **GPU**: `Runtime → Change runtime type → T4 GPU`.

## 0. Setup — GPU, clone repo, install dependencies

In [ ]:
# Cek GPU
import torch
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  GPU tidak aktif. Runtime → Change runtime type → T4 GPU')


In [ ]:
# Clone repo (kode saja — data menyusul lewat Google Drive)
import os
REPO_URL = 'https://github.com/yumikopubangelo/duck_egg_fertility_detection.git'
PROJ = '/content/duck_egg_fertility_detection'
if not os.path.exists(PROJ):
    !git clone -b pipeline {REPO_URL} {PROJ}
%cd {PROJ}
print('CWD:', os.getcwd())


In [ ]:
# Dependencies (Colab sudah punya torch/opencv/sklearn; ini melengkapi)
!pip -q install pyyaml scikit-image >/dev/null 2>&1
import cv2, numpy as np, sklearn, matplotlib
print('cv2', cv2.__version__, '| sklearn', sklearn.__version__)


## 1. Data — mount Google Drive

Dataset gambar **tidak** ada di repo (gitignored). Siapkan di Google Drive dengan struktur:

```
MyDrive/duck_egg_data/
├── train/{fertile,infertile}/*.jpg
├── val/{fertile,infertile}/*.jpg
└── test/{fertile,infertile}/*.jpg
```

Ubah `DRIVE_DATA` di bawah sesuai lokasi folder Anda. Sel ini menyalin data ke `data/` di runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE_DATA = '/content/drive/MyDrive/duck_egg_data'   # ← sesuaikan
assert os.path.exists(DRIVE_DATA), f'Folder tidak ditemukan: {DRIVE_DATA}'

for split in ['train','val','test']:
    src = os.path.join(DRIVE_DATA, split)
    dst = os.path.join(PROJ, 'data', split)
    if os.path.exists(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = sum(len(files) for _,_,files in os.walk(dst))
        print(f'{split}: {n} file disalin → {dst}')
    else:
        print(f'⚠️  {src} tidak ada')


In [ ]:
# Verifikasi jumlah per kelas
import glob
for split in ['train','val','test']:
    for cls in ['fertile','infertile']:
        n = len(glob.glob(f'data/{split}/{cls}/*.jpg')) + len(glob.glob(f'data/{split}/{cls}/*.jpeg'))
        print(f'{split}/{cls}: {n}')


## 2. Generate masks (3 kelas: BG / Vaskular / Embrio)

Mask dibuat otomatis via heuristik morfologi ([scripts/03b_generate_masks.py](scripts/03b_generate_masks.py)):
- **Vaskular (kelas 1)** = black-hat morphology
- **Embrio (kelas 2)** = region gelap (mean − k·std) di dalam telur
- **Infertil** = mask kosong (semua background)

> 📌 Catatan kejujuran: ini *pseudo-label* (bukan anotasi pakar). U-Net belajar meniru heuristik ini. Sebutkan sebagai **weak supervision / morphological pseudo-labels** di artikel.

In [ ]:
# k lebih kecil = embrio lebih sensitif. min-embryo-area mencegah noise.
!python scripts/03b_generate_masks.py \
    --data-dir ./data --output-dir ./data/segmentation \
    --splits train val test \
    --embryo-k 0.5 --min-embryo-area 0.01 \
    --mask-format multiclass --visualize

# Cek distribusi kelas pada mask
import glob, cv2, numpy as np
fr_all = {0:0,1:0,2:0}; tot=0
for m in glob.glob('data/segmentation/train/masks/*.png'):
    mm = cv2.imread(m, cv2.IMREAD_UNCHANGED)
    for k in (0,1,2): fr_all[k]+=int((mm==k).sum())
    tot += mm.size
print('Distribusi kelas (train):', {k: round(v/tot*100,2) for k,v in fr_all.items()}, '%')


## 3. Train U-Net 3 kelas — **loss anti-collapse**

Inti perbaikan: **Weighted Cross-Entropy + Multiclass Dice**.
- *Class weight* dari inverse-frequency (di-cap) → kelas minoritas (vaskular/embrio) tidak diabaikan.
- *Dice loss* multiclass → robust terhadap imbalance ekstrem.

Loss lama (`ce` polos) membuat model collapse ke background. Ini menggantinya.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, cv2, glob, os, time
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import sys; sys.path.append(PROJ)
from src.segmentation.unet_lightweight import create_unet_lightweight

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE, N_CLASSES = 256, 3

# ── Dataset (image + multiclass mask) ──
class SegDS(Dataset):
    def __init__(self, split, augment=False):
        self.augment = augment
        idir = f'data/segmentation/{split}/images'
        mdir = f'data/segmentation/{split}/masks'
        self.pairs = []
        for ip in sorted(glob.glob(f'{idir}/*')):
            stem = os.path.splitext(os.path.basename(ip))[0]
            mp = f'{mdir}/{stem}.png'
            if os.path.exists(mp): self.pairs.append((ip, mp))
        self.tt = T.ToTensor()
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        img = Image.open(ip).convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR)
        msk = Image.open(mp).convert('L').resize((IMG_SIZE,IMG_SIZE), Image.NEAREST)
        img = np.array(img); msk = np.array(msk).astype(np.int64)
        if self.augment and np.random.rand() < 0.5:        # horizontal flip
            img = img[:, ::-1].copy(); msk = msk[:, ::-1].copy()
        if self.augment and np.random.rand() < 0.3:        # vertical flip
            img = img[::-1].copy(); msk = msk[::-1].copy()
        x = self.tt(Image.fromarray(img))
        y = torch.from_numpy(msk).long()
        return x, y

train_ds, val_ds = SegDS('train', augment=True), SegDS('val')
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)
print(f'train {len(train_ds)} | val {len(val_ds)} | device {DEVICE}')


In [ ]:
# ── Class weights dari frekuensi piksel (inverse-freq, di-cap) ──
counts = np.zeros(N_CLASSES, dtype=np.float64)
for _, y in train_ds:
    for k in range(N_CLASSES): counts[k] += (y==k).sum().item()
freq = counts / counts.sum()
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * N_CLASSES
weights = np.clip(weights, 0.3, 8.0)            # cap supaya tidak ekstrem
class_w = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print('Freq kelas :', freq.round(4))
print('Class weight:', class_w.cpu().numpy().round(3))

# ── Multiclass Dice loss ──
def dice_loss(logits, target, eps=1.0):
    probs = F.softmax(logits, dim=1)
    t1h = F.one_hot(target, N_CLASSES).permute(0,3,1,2).float()
    dims = (0,2,3)
    inter = (probs*t1h).sum(dims); card = probs.sum(dims)+t1h.sum(dims)
    dice = (2*inter+eps)/(card+eps)
    return 1 - dice.mean()

ce = nn.CrossEntropyLoss(weight=class_w)
def criterion(logits, target):
    return ce(logits, target) + dice_loss(logits, target)

def iou_score(logits, target):
    pred = torch.argmax(logits,1)
    ious=[]
    for k in range(N_CLASSES):
        p,t = (pred==k), (target==k)
        inter=(p&t).sum().item(); union=(p|t).sum().item()
        if union>0: ious.append(inter/union)
    return float(np.mean(ious)) if ious else 0.0


In [ ]:
# ── Training loop ──
torch.manual_seed(42); np.random.seed(42)
net = create_unet_lightweight(n_channels=3, n_classes=N_CLASSES, bilinear=True, dropout_rate=0.2).to(DEVICE)
opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=6)

EPOCHS = 80
hist = {'train_loss':[], 'val_loss':[], 'val_iou':[], 'val_vasc_iou':[], 'val_emb_iou':[]}
best_iou, best_state = -1, None
t0=time.time()
for ep in range(1, EPOCHS+1):
    net.train(); tl=0
    for x,y in train_dl:
        x,y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); out=net(x); loss=criterion(out,y); loss.backward(); opt.step()
        tl += loss.item()*x.size(0)
    tl/=len(train_ds)
    # validate
    net.eval(); vl=0; ious=[]; per=[[],[]]
    with torch.no_grad():
        for x,y in val_dl:
            x,y=x.to(DEVICE),y.to(DEVICE); out=net(x)
            vl += criterion(out,y).item()*x.size(0); ious.append(iou_score(out,y))
            pred=torch.argmax(out,1)
            for ci,k in enumerate([1,2]):
                p,t=(pred==k),(y==k); u=(p|t).sum().item()
                if u>0: per[ci].append((p&t).sum().item()/u)
    vl/=len(val_ds); viou=float(np.mean(ious))
    vv=float(np.mean(per[0])) if per[0] else 0.0
    ve=float(np.mean(per[1])) if per[1] else 0.0
    sched.step(viou)
    hist['train_loss'].append(tl); hist['val_loss'].append(vl)
    hist['val_iou'].append(viou); hist['val_vasc_iou'].append(vv); hist['val_emb_iou'].append(ve)
    if viou>best_iou: best_iou=viou; best_state={k:v.cpu().clone() for k,v in net.state_dict().items()}
    if ep%5==0 or ep==1:
        print(f'ep {ep:3d} | train {tl:.3f} | val {vl:.3f} | IoU {viou:.3f} | vasc {vv:.3f} emb {ve:.3f}')
print(f'Selesai {time.time()-t0:.0f}s | best val IoU {best_iou:.3f}')


In [ ]:
# ── Simpan model (format kompatibel dengan pipeline: model_state_dict + config) ──
os.makedirs('models/unet', exist_ok=True)
# backup model lama
if os.path.exists('models/unet/model.pth'):
    shutil.copy('models/unet/model.pth', 'models/unet/model_OLD_collapsed.pth')
net.load_state_dict(best_state)
ckpt = {
    'model_state_dict': net.state_dict(),
    'epoch': EPOCHS,
    'metrics': {'val_iou': best_iou},
    'config': {'model': {'lightweight':True,'n_channels':3,'n_classes':3,
                         'bilinear':True,'dropout_rate':0.2}},
}
torch.save(ckpt, 'models/unet/model.pth')
print('Tersimpan: models/unet/model.pth (lama → model_OLD_collapsed.pth)')


In [ ]:
# ── Kurva training ──
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(13,4.2))
ep=range(1,len(hist['train_loss'])+1)
ax[0].plot(ep,hist['train_loss'],label='train'); ax[0].plot(ep,hist['val_loss'],label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep,hist['val_iou'],label='mean IoU',lw=2)
ax[1].plot(ep,hist['val_vasc_iou'],label='vaskular IoU',ls='--')
ax[1].plot(ep,hist['val_emb_iou'],label='embrio IoU',ls='--')
ax[1].set_title('Validation IoU'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('docs/fig_unet_training_curves.png',dpi=130,bbox_inches='tight'); plt.show()
print('✅ Kalau vaskular/embrio IoU > 0 dan naik → tidak collapse lagi.')


## 4. Verifikasi U-Net — **pastikan tidak collapse**

Prediksi pada test set. Kalau vaskular (oranye) & embrio (hijau) muncul di telur fertil dan bersih di infertil → berhasil. Kalau masih background semua → naikkan `EPOCHS` atau class weight.

In [ ]:
import torch, numpy as np, glob, os
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt, matplotlib.patches as mpatches

net.eval(); tt=T.ToTensor()
def predict(path):
    pil=Image.open(path).convert('RGB').resize((256,256),Image.BILINEAR)
    with torch.no_grad():
        pred=torch.argmax(net(tt(pil).unsqueeze(0).to(DEVICE)),1)[0].cpu().numpy().astype(np.uint8)
    return np.array(pil), pred
def colorize(p):
    o=np.zeros((*p.shape,3),np.uint8); o[p==0]=[40,90,160]; o[p==1]=[230,120,30]; o[p==2]=[40,180,60]; return o
def overlay(rgb,p):
    o=rgb.copy()
    o[p==1]=(o[p==1]*0.45+np.array([230,120,30])*0.55).astype(np.uint8)
    o[p==2]=(o[p==2]*0.45+np.array([40,180,60])*0.55).astype(np.uint8); return o

# ambil 2 fertil + 2 infertil dari test
samples=[]
for cls,lab in [('fertile','FERTIL'),('infertile','INFERTIL')]:
    fs=sorted(glob.glob(f'data/test/{cls}/*.jpg'))[:2]
    for f in fs: samples.append((lab,f))

fig,axes=plt.subplots(len(samples),3,figsize=(11,3.4*len(samples))); fig.patch.set_facecolor('white')
fig.suptitle('U-Net Baru — Output Segmentasi (BG / Vaskular oranye / Embrio hijau)',fontsize=13,fontweight='bold')
COLT=['Original','Segmentation Mask','Overlay']
for ri,(lab,f) in enumerate(samples):
    rgb,pred=predict(f)
    vasc=(pred==1).mean()*100; emb=(pred==2).mean()*100
    print(f'{lab:9s} {os.path.basename(f):16s} vaskular={vasc:.1f}% embrio={emb:.1f}% kelas={np.unique(pred)}')
    rcol='#1E8449' if lab=='FERTIL' else '#C0392B'
    for ci,im in enumerate([rgb,colorize(pred),overlay(rgb,pred)]):
        ax=axes[ri,ci]; ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_color(rcol); sp.set_linewidth(3)
        if ri==0: ax.set_title(COLT[ci],fontsize=11,fontweight='bold')
        if ci==0: ax.set_ylabel(f'{lab}\nV={vasc:.0f}% E={emb:.0f}%',fontsize=10,fontweight='bold',color=rcol)
fig.legend(handles=[mpatches.Patch(color=(40/255,90/255,160/255),label='Background'),
                    mpatches.Patch(color=(230/255,120/255,30/255),label='Vaskular'),
                    mpatches.Patch(color=(40/255,180/255,60/255),label='Embrio')],
           loc='lower center',ncol=3,fontsize=10,bbox_to_anchor=(0.5,-0.01))
plt.tight_layout(rect=[0,0.03,1,0.96]); plt.savefig('docs/fig_unet_segmentation_output.png',dpi=130,bbox_inches='tight'); plt.show()


## 5. Ekstraksi fitur (classical + deep embedding U-Net baru)

Fitur 322-dim untuk AWC. Deep embedding diambil dari **bottleneck U-Net yang baru dilatih**.

In [ ]:
# Pakai checkpoint U-Net baru untuk deep embedding
!python scripts/04_extract_features.py \
    --data-root data --output-dir data/features \
    --mode hybrid --unet-checkpoint models/unet/model.pth

import numpy as np
Xtr=np.load('data/features/awc_features.npy'); ytr=np.load('data/features/awc_labels.npy')
Xte=np.load('data/features/awc_test_features.npy'); yte=np.load('data/features/awc_test_labels.npy')
print('Train:',Xtr.shape,'Test:',Xte.shape)
print('Label train (0=infertil,1=fertil):', np.bincount(ytr))
print('Ada NaN?', np.isnan(Xtr).any(), '| Inf?', np.isinf(Xtr).any())


## 6. Train AWC + evaluasi

AWC dilatih ulang dari fitur baru. Sel diberi penanganan error agar masalah (NaN, shape, label) langsung terlihat.

In [ ]:
# Bersihkan NaN/Inf bila ada (penyebab error AWC paling umum)
import numpy as np
Xtr=np.load('data/features/awc_features.npy'); Xte=np.load('data/features/awc_test_features.npy')
Xtr=np.nan_to_num(Xtr,nan=0.0,posinf=0.0,neginf=0.0)
Xte=np.nan_to_num(Xte,nan=0.0,posinf=0.0,neginf=0.0)
np.save('data/features/awc_features.npy',Xtr); np.save('data/features/awc_test_features.npy',Xte)
print('Fitur dibersihkan. Train',Xtr.shape,'Test',Xte.shape)


In [ ]:
# Latih AWC lewat script resmi (ANOVA k=20, sesuai configs/awc_config.yaml)
!python scripts/05_train_awc.py --config configs/awc_config.yaml

import json
with open('results/awc_evaluation/metrics.json') as f: m=json.load(f)
print('=== Evaluasi AWC ===')
for k,v in m['evaluation'].items(): print(f'  {k}: {v:.4f}')


In [ ]:
# Akurasi & confusion matrix (mapping cluster→label terbaik)
import numpy as np, pickle
from itertools import permutations
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from src.clustering.awc import AdaptiveWeightedClustering

model = AdaptiveWeightedClustering.load('models/awc/awc_model.pkl')
Xte=np.load('data/features/awc_test_features.npy'); yte=np.load('data/features/awc_test_labels.npy')
pred=model.predict(Xte)
# map cluster id -> label
best_acc,best=-1,pred
for perm in permutations(range(len(np.unique(pred)))):
    mp=np.array([perm[p] for p in pred])
    a=accuracy_score(yte,mp)
    if a>best_acc: best_acc,best=a,mp
print(f'Akurasi test: {best_acc:.3f} | F1: {f1_score(yte,best,average="macro"):.3f}')

import matplotlib.pyplot as plt
cm=confusion_matrix(yte,best)
fig,ax=plt.subplots(figsize=(4.5,4))
im=ax.imshow(cm,cmap='Reds')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=14,fontweight='bold',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Infertil','Fertil']); ax.set_yticklabels(['Infertil','Fertil'])
ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual'); ax.set_title(f'AWC Confusion (acc={best_acc:.1%})')
plt.tight_layout(); plt.savefig('docs/fig_awc_confusion_retrained.png',dpi=130,bbox_inches='tight'); plt.show()


In [ ]:
# Feature importance AWC (bobot adaptif per fitur terpilih)
import numpy as np, matplotlib.pyplot as plt
info = model.get_cluster_info()
fi = np.array(info['feature_importance'])
idx = np.argsort(fi)[::-1][:15]
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(range(len(idx)), fi[idx], color='#C0392B', edgecolor='black')
ax.set_xticks(range(len(idx))); ax.set_xticklabels([f'f{i}' for i in idx], rotation=45)
ax.set_ylabel('Importance'); ax.set_title('AWC — 15 Fitur Terpenting (bobot adaptif)')
ax.grid(axis='y',alpha=.3)
plt.tight_layout(); plt.savefig('docs/fig_awc_feature_importance_retrained.png',dpi=130,bbox_inches='tight'); plt.show()
print('Silhouette:',info.get('silhouette_score'),'| iterasi:',info.get('iterations'))


## 7. Simpan hasil ke Drive

Salin model & figure baru kembali ke Google Drive agar tidak hilang saat runtime mati.

In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/duck_egg_retrained'
os.makedirs(OUT, exist_ok=True)
for p in ['models/unet/model.pth','models/awc/awc_model.pkl',
          'results/awc_evaluation/metrics.json']:
    if os.path.exists(p): shutil.copy(p, OUT); print('✓',p)
# semua figure
os.makedirs(f'{OUT}/figures', exist_ok=True)
import glob
for f in glob.glob('docs/fig_*.png'):
    shutil.copy(f, f'{OUT}/figures');
print('Figure & model tersimpan di', OUT)


---
### ✅ Ringkasan
- **U-Net** dilatih ulang dengan **weighted CE + Dice** → tidak collapse (cek IoU vaskular/embrio > 0).
- **AWC** dilatih ulang dari fitur baru, dengan pembersihan NaN/Inf untuk cegah error.
- Semua figure ada di `docs/` dan disalin ke Drive.

> 📝 **Untuk artikel:** sebutkan mask vaskular/embrio sebagai *morphological pseudo-labels (weak supervision)*. Novelty utama tetap **AWC** pada vektor fitur 322-dim.